# Machine Condition Classification — Logistic Regression Pipeline

## 1. Imports

In [19]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# from google.colab import drive
# drive.mount('/content/drive')

## 2. Load Data

In [20]:
TRAIN_PATH      = 'Classification_Dataset/student_dataset.csv'
TEST_PATH       = 'Classification_Dataset/test.csv'

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f'Train shape : {df_train.shape}')
print(f'Test shape  : {df_test.shape}')
print(f'\nClass distribution (train):')
print(df_train['condition'].value_counts().sort_index())
print(f'\nTest nulls per column:')
print(df_test.isnull().sum())

Train shape : (16000, 13)
Test shape  : (4000, 12)

Class distribution (train):
condition
0    8000
1    4000
2    4000
Name: count, dtype: int64

Test nulls per column:
unit_id             0
unit_type          11
ambient_temp       10
process_temp       13
coolant_flow       12
shaft_speed         8
drive_torque       12
power_load          8
component_wear     15
thermal_stress     16
mechanical_load     3
vibration_index    15
dtype: int64


## 3. Feature Engineering

Key findings from error analysis:
- Thermal vs Normal is the hardest boundary → features: process_temp, thermal_stress, heat_index
- Mechanical vs Normal → features: component_wear (biggest gap: +78), vibration_index, mechanical_load
- Log-transforms help right-skewed multiplicative features
- Squared + cubic terms capture non-linear class thresholds

In [21]:
# Apply all feature engineering to a raw dataframe.
def make_features(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()

    # ── Encode categorical unit_type (S / M / H) 
    le = LabelEncoder()
    df['unit_type_enc'] = le.fit_transform(df['unit_type'].fillna('Unknown'))

    # ── Thermal features
    df['temp_diff']          = df['process_temp'] - df['ambient_temp']
    df['temp_ratio']         = df['process_temp'] / (df['ambient_temp'] + 1)
    df['thermal_x_stress']   = df['thermal_stress'] * df['process_temp']
    df['stress_per_coolant'] = df['thermal_stress'] / (df['coolant_flow'] + 1)
    df['coolant_efficiency'] = df['coolant_flow'] / (df['process_temp'] + 1)
    df['heat_index']         = df['process_temp'] * df['thermal_stress'] / (df['coolant_flow'] + 1)
    df['total_stress']       = df['thermal_stress'] + df['mechanical_load']

    # ── Mechanical features
    df['torque_speed']         = df['drive_torque'] * df['shaft_speed']
    df['wear_load']            = df['component_wear'] * df['mechanical_load']
    df['vibration_torque']     = df['vibration_index'] * df['drive_torque']
    df['power_per_speed']      = df['power_load'] / (df['shaft_speed'] + 1)
    df['wear_speed']           = df['component_wear'] * df['shaft_speed']
    df['vibration_wear']       = df['vibration_index'] * df['component_wear']
    df['mechanical_vibration'] = df['mechanical_load'] * df['vibration_index']
    df['torque_wear']          = df['drive_torque'] * df['component_wear']
    df['power_torque_ratio']   = df['power_load'] / (df['drive_torque'] + 1)
    df['speed_torque_power']   = df['shaft_speed'] * df['drive_torque'] / (df['power_load'] + 1)

    # ── Log transforms (right-skewed multiplicative features)
    log_cols = [
        'component_wear', 'torque_speed', 'wear_load', 'vibration_torque',
        'wear_speed', 'vibration_wear', 'mechanical_vibration', 'torque_wear', 'heat_index'
    ]
    for col in log_cols:
        df[f'log_{col}'] = np.log1p(np.abs(df[col].fillna(0)))

    # ── Squared terms (non-linear class thresholds) 
    sq_cols = ['thermal_stress', 'mechanical_load', 'vibration_index', 'component_wear']
    for col in sq_cols:
        df[f'sq_{col}'] = df[col] ** 2

    # ── Cubic terms (top discriminative features) ──
    cb_cols = ['thermal_stress', 'mechanical_load', 'vibration_index']
    for col in cb_cols:
        df[f'cb_{col}'] = df[col] ** 3

    # ── Triple interaction terms
    df['thermal_mech_vib']  = df['thermal_stress'] * df['mechanical_load'] * df['vibration_index']
    df['wear_torque_speed'] = df['component_wear'] * df['drive_torque'] * df['shaft_speed']

    return df


EXCLUDE_COLS = {'unit_id', 'unit_type', 'condition'}

df_tr = make_features(df_train)
df_te = make_features(df_test)

feat_cols = [c for c in df_tr.columns if c not in EXCLUDE_COLS]
print(f'Total engineered features: {len(feat_cols)}')
print(feat_cols)

Total engineered features: 46
['ambient_temp', 'process_temp', 'coolant_flow', 'shaft_speed', 'drive_torque', 'power_load', 'component_wear', 'thermal_stress', 'mechanical_load', 'vibration_index', 'unit_type_enc', 'temp_diff', 'temp_ratio', 'thermal_x_stress', 'stress_per_coolant', 'coolant_efficiency', 'heat_index', 'total_stress', 'torque_speed', 'wear_load', 'vibration_torque', 'power_per_speed', 'wear_speed', 'vibration_wear', 'mechanical_vibration', 'torque_wear', 'power_torque_ratio', 'speed_torque_power', 'log_component_wear', 'log_torque_speed', 'log_wear_load', 'log_vibration_torque', 'log_wear_speed', 'log_vibration_wear', 'log_mechanical_vibration', 'log_torque_wear', 'log_heat_index', 'sq_thermal_stress', 'sq_mechanical_load', 'sq_vibration_index', 'sq_component_wear', 'cb_thermal_stress', 'cb_mechanical_load', 'cb_vibration_index', 'thermal_mech_vib', 'wear_torque_speed']


## 4. Prepare Feature Matrices

In [22]:
X_train = df_tr[feat_cols]
y_train = df_tr['condition'].values
X_test  = df_te[feat_cols]

print(f'X_train shape : {X_train.shape}')
print(f'X_test shape  : {X_test.shape}')
print(f'y_train distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}')

X_train shape : (16000, 46)
X_test shape  : (4000, 46)
y_train distribution: {np.int64(0): np.int64(8000), np.int64(1): np.int64(4000), np.int64(2): np.int64(4000)}


## 5. Cross-Validation (Best Pipeline)

Key hyperparameter decisions:

- Penalty: L2 Beats L1 and ElasticNet on this data.
- Solver: lbfgs Best for L2 multiclass.
- C = 0.3 Tuned via fine grid search.
- class_weight: None. Natural 2:1:1 ratio works better than 'balanced'.
- Poly degree: 2, interaction_only Biggest single accuracy boost (+6%).
- Imputer: median. More confident than iterative on this data.

In [23]:
best_pipeline = Pipeline([
    ('imp',  SimpleImputer(strategy='median')),
    ('sc',   StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('sc2',  StandardScaler()),
    ('clf',  LogisticRegression(
        penalty='l2',
        solver='lbfgs',
        C=0.3,
        max_iter=5000,
        class_weight=None,
        random_state=42
    ))
])

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(best_pipeline, X_train, y_train, cv=cv5, scoring='accuracy', n_jobs=-1)

print('=' * 50)
print('5-Fold CV Results')
print('=' * 50)
print(f'Mean Accuracy : {scores.mean():.4f}')
print(f'Std           : {scores.std():.4f}')
print(f'Min / Max     : {scores.min():.4f} / {scores.max():.4f}')
print(f'Per fold      : {[round(s, 4) for s in scores]}')

5-Fold CV Results
Mean Accuracy : 0.8906
Std           : 0.0022
Min / Max     : 0.8884 / 0.8947
Per fold      : [np.float64(0.8884), np.float64(0.89), np.float64(0.8906), np.float64(0.8947), np.float64(0.8891)]


## 6. Holdout Evaluation (80/20 Split)

In [24]:
X_tr_h, X_va_h, y_tr_h, y_va_h = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

best_pipeline.fit(X_tr_h, y_tr_h)
y_pred_h = best_pipeline.predict(X_va_h)

print(f'Holdout Accuracy : {accuracy_score(y_va_h, y_pred_h):.4f}\n')

print('Classification Report:')
print(classification_report(
    y_va_h, y_pred_h,
    target_names=['Normal (0)', 'Thermal Fail (1)', 'Mechanical Fail (2)']
))

print('Confusion Matrix (row=actual, col=predicted):')
cm = confusion_matrix(y_va_h, y_pred_h)
cm_df = pd.DataFrame(cm,
    index=['Actual Normal', 'Actual Thermal', 'Actual Mechanic'],
    columns=['Pred Normal', 'Pred Thermal', 'Pred Mechanic'])
print(cm_df)

Holdout Accuracy : 0.8975

Classification Report:
                     precision    recall  f1-score   support

         Normal (0)       0.90      0.94      0.92      1600
   Thermal Fail (1)       0.88      0.85      0.86       800
Mechanical Fail (2)       0.91      0.86      0.88       800

           accuracy                           0.90      3200
          macro avg       0.90      0.88      0.89      3200
       weighted avg       0.90      0.90      0.90      3200

Confusion Matrix (row=actual, col=predicted):
                 Pred Normal  Pred Thermal  Pred Mechanic
Actual Normal           1503            63             34
Actual Thermal            86           683             31
Actual Mechanic           80            34            686


## 7. Preprocessing Helper

We preprocess once and reuse the fitted transformers across models (faster for ensemble building).

In [25]:
def preprocess(X_tr, X_te, imputer_type='median'):

    if imputer_type == 'median':
        imp = SimpleImputer(strategy='median')
    else:
        imp = IterativeImputer(max_iter=5, random_state=42)

    X_tr_imp = imp.fit_transform(X_tr)
    X_te_imp = imp.transform(X_te)

    sc1  = StandardScaler().fit(X_tr_imp)
    X_tr_sc = sc1.transform(X_tr_imp)
    X_te_sc = sc1.transform(X_te_imp)

    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
    X_tr_p = poly.fit_transform(X_tr_sc)
    X_te_p = poly.transform(X_te_sc)

    sc2  = StandardScaler().fit(X_tr_p)
    X_tr_f = sc2.transform(X_tr_p)
    X_te_f = sc2.transform(X_te_p)

    return X_tr_f, X_te_f


print('Preprocessing with median imputer (used for single-model submission)')
X_tr_f, X_te_f = preprocess(X_train, X_test, imputer_type='median')
print(f'Shape after poly expansion: {X_tr_f.shape}  (train)  {X_te_f.shape}  (test)')

Preprocessing with median imputer (used for single-model submission)
Shape after poly expansion: (16000, 1081)  (train)  (4000, 1081)  (test)


## 8. Submission 1 — Single Best Model

Train on full dataset with C=0.3, median imputer. Highest mean confidence (89.13%).

In [26]:
clf_single = LogisticRegression(
    penalty='l2', solver='lbfgs', C=0.3, max_iter=5000, random_state=42
)
clf_single.fit(X_tr_f, y_train)

train_acc = accuracy_score(y_train, clf_single.predict(X_tr_f))
print(f'Full-train accuracy : {train_acc:.4f}')

preds_single = clf_single.predict(X_te_f)
probs_single = clf_single.predict_proba(X_te_f)

print(f'Mean confidence     : {probs_single.max(1).mean():.4f}')
print(f'Low-conf (<0.5)     : {(probs_single.max(1) < 0.5).sum()} predictions')
print(f'\nPrediction distribution:')
print(pd.Series(preds_single).value_counts().sort_index()
      .rename({0: 'Normal (0)', 1: 'Thermal (1)', 2: 'Mechanical (2)'}))

sub_single = pd.DataFrame({'unit_id': df_test['unit_id'], 'condition': preds_single})
sub_single.to_csv('Classification_Dataset/submission_single_C03.csv', index=False)
print('\nSaved: submission_single_C03.csv')
sub_single.head()

Full-train accuracy : 0.9038
Mean confidence     : 0.8911
Low-conf (<0.5)     : 67 predictions

Prediction distribution:
Normal (0)        2114
Thermal (1)        936
Mechanical (2)     950
Name: count, dtype: int64

Saved: submission_single_C03.csv


,unit_id,condition
0,UNT-184630,0
1,UNT-228473,0
2,UNT-255783,1
3,UNT-119059,2
4,UNT-205559,0


## 9. Submission 2 — Ensemble Model

Averages probabilities from 3 diverse models:
1. Median imputer, C=0.2
2. Median imputer, C=0.3  
3. Iterative (MICE) imputer, C=0.3

Differs from single model on 18 predictions — more robust on the 60 null rows in test.

In [27]:
def get_test_probs(X_tr_f, X_te_f, y, C=0.3):

    clf = LogisticRegression(penalty='l2', solver='lbfgs', C=C,
                              max_iter=5000, random_state=42)
    clf.fit(X_tr_f, y)
    return clf.predict_proba(X_te_f)


print('Preprocessing with median imputer...')
X_tr_med, X_te_med = preprocess(X_train, X_test, imputer_type='median')

print('Preprocessing with iterative imputer (MICE)...')
X_tr_iter, X_te_iter = preprocess(X_train, X_test, imputer_type='iterative')

print('\nBuilding 3 ensemble components...')
p1 = get_test_probs(X_tr_med,  X_te_med,  y_train, C=0.2)  # median, C=0.2
print('  [1/3] Median C=0.2  done')
p2 = get_test_probs(X_tr_med,  X_te_med,  y_train, C=0.3)  # median, C=0.3
print('  [2/3] Median C=0.3  done')
p3 = get_test_probs(X_tr_iter, X_te_iter, y_train, C=0.3)  # iterative, C=0.3
print('  [3/3] Iterative C=0.3 done')

avg_probs  = (p1 + p2 + p3) / 3
preds_ens  = avg_probs.argmax(1)

print(f'\nEnsemble mean confidence : {avg_probs.max(1).mean():.4f}')
print(f'Differences vs single    : {(preds_ens != preds_single).sum()} predictions')
print(f'\nEnsemble prediction distribution:')
print(pd.Series(preds_ens).value_counts().sort_index()
      .rename({0: 'Normal (0)', 1: 'Thermal (1)', 2: 'Mechanical (2)'}))

sub_ens = pd.DataFrame({'unit_id': df_test['unit_id'], 'condition': preds_ens})
sub_ens.to_csv('Classification_Dataset/submission_best_ensemble.csv', index=False)
print('\nSaved: submission_best_ensemble.csv')
sub_ens.head()

Preprocessing with median imputer...
Preprocessing with iterative imputer (MICE)...

Building 3 ensemble components...
  [1/3] Median C=0.2  done
  [2/3] Median C=0.3  done
  [3/3] Iterative C=0.3 done

Ensemble mean confidence : 0.8889
Differences vs single    : 19 predictions

Ensemble prediction distribution:
Normal (0)        2113
Thermal (1)        938
Mechanical (2)     949
Name: count, dtype: int64

Saved: submission_best_ensemble.csv


,unit_id,condition
0,UNT-184630,0
1,UNT-228473,0
2,UNT-255783,1
3,UNT-119059,2
4,UNT-205559,0
